# Model 3 V2 — Group-Aware Data Audit

This audit prepares CIC-IDS2017 for source-file-aware validation. It does **not** train or overwrite any model.

Goals:
- discover all CIC-IDS2017 CSV groups
- normalize labels
- report rows, benign/attack counts and attack families per source file
- detect missing/infinite values and duplicate rows
- create a source-group manifest used by the V2 training notebook

In [3]:
from pathlib import Path
import json, re
import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd()
while PROJECT_DIR.name != "EVIDENT" and PROJECT_DIR.parent != PROJECT_DIR:
    PROJECT_DIR = PROJECT_DIR.parent
if PROJECT_DIR.name != "EVIDENT":
    PROJECT_DIR = Path(r"C:/Users/bimal/OneDrive/Desktop/EVIDENT")

RAW_DIR = PROJECT_DIR / "data/raw/behavior"
META_DIR = PROJECT_DIR / "models/behavior/metadata"
META_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = PROJECT_DIR / "configs/model3_v2_group_aware.json"

print("Project:", PROJECT_DIR)
print("Raw data:", RAW_DIR)

Project: c:\Users\bimal\OneDrive\Desktop\EVIDENT
Raw data: c:\Users\bimal\OneDrive\Desktop\EVIDENT\data\raw\behavior


In [6]:
PROJECT_DIR = Path(r"C:\Users\bimal\OneDrive\Desktop\EVIDENT")

RAW_DIR = PROJECT_DIR / "data" / "raw" / "behaviour"

if not RAW_DIR.exists():
    raise FileNotFoundError(f"Dataset directory not found: {RAW_DIR}")

print("Project:", PROJECT_DIR)
print("Dataset directory:", RAW_DIR)

Project: C:\Users\bimal\OneDrive\Desktop\EVIDENT
Dataset directory: C:\Users\bimal\OneDrive\Desktop\EVIDENT\data\raw\behaviour


In [7]:
# Discover CSV groups
files = sorted(RAW_DIR.glob("*.csv"))

if not files:
    raise FileNotFoundError(
        f"No CIC-IDS2017 CSV files found in {RAW_DIR}"
    )

print("Available CSV groups:")

for f in files:
    print(" -", f.name)

print("Total:", len(files))

Available CSV groups:
 - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
 - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
 - Friday-WorkingHours-Morning.pcap_ISCX.csv
 - Monday-WorkingHours.pcap_ISCX.csv
 - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
 - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
 - Tuesday-WorkingHours.pcap_ISCX.csv
 - Wednesday-workingHours.pcap_ISCX.csv
Total: 8


In [8]:
def find_label_column(columns):
    for c in columns:
        if str(c).strip().lower() == "label":
            return c
    raise ValueError("Could not find Label column")

rows=[]
for path in files:
    header=pd.read_csv(path,nrows=0)
    label_col=find_label_column(header.columns)
    s=pd.read_csv(path,usecols=[label_col],low_memory=False)[label_col].astype(str).str.strip()
    vc=s.value_counts(dropna=False)
    attacks=vc[vc.index.str.upper()!="BENIGN"]
    rows.append({
        "file":path.name,
        "rows":int(len(s)),
        "benign_rows":int((s.str.upper()=="BENIGN").sum()),
        "attack_rows":int((s.str.upper()!="BENIGN").sum()),
        "attack_ratio":float((s.str.upper()!="BENIGN").mean()),
        "original_labels":sorted(map(str, s.unique().tolist())),
        "label_counts":{str(k):int(v) for k,v in vc.items()}
    })
audit=pd.DataFrame(rows)
display(audit[["file","rows","benign_rows","attack_rows","attack_ratio","original_labels"]])

,file,rows,benign_rows,attack_rows,attack_ratio,original_labels
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,97718,128027,0.567131,"[BENIGN, DDoS]"
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,127537,158930,0.554793,"[BENIGN, PortScan]"
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,189067,1966,0.010291,"[BENIGN, Bot]"
3,Monday-WorkingHours.pcap_ISCX.csv,529918,529918,0,0.000000,[BENIGN]
4,Thursday-WorkingHours-Afternoon-Infilteration....,288602,288566,36,0.000125,"[BENIGN, Infiltration]"
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,170366,168186,2180,0.012796,"[BENIGN, Web Attack � Brute Force, Web Attack ..."
6,Tuesday-WorkingHours.pcap_ISCX.csv,445909,432074,13835,0.031027,"[BENIGN, FTP-Patator, SSH-Patator]"
7,Wednesday-workingHours.pcap_ISCX.csv,692703,440031,252672,0.364762,"[BENIGN, DoS GoldenEye, DoS Hulk, DoS Slowhttp..."


In [9]:
# Feature-level integrity audit on each file
integrity=[]
for path in files:
    df=pd.read_csv(path,low_memory=False)
    label_col=find_label_column(df.columns)
    X=df.drop(columns=[label_col])
    numeric=X.select_dtypes(include=[np.number])
    integrity.append({
        "file":path.name,
        "rows":len(df),
        "columns":len(df.columns),
        "numeric_columns":numeric.shape[1],
        "missing_cells":int(numeric.isna().sum().sum()),
        "infinite_cells":int(np.isinf(numeric.to_numpy(dtype=float,copy=False)).sum()),
        "exact_duplicate_rows":int(df.duplicated().sum())
    })
integrity_df=pd.DataFrame(integrity)
display(integrity_df)

,file,rows,columns,numeric_columns,missing_cells,infinite_cells,exact_duplicate_rows
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,79,78,4,64,2633
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,79,78,15,727,72353
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,79,78,28,216,6888
3,Monday-WorkingHours.pcap_ISCX.csv,529918,79,78,64,810,26935
4,Thursday-WorkingHours-Afternoon-Infilteration....,288602,79,78,18,396,35630
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,170366,79,78,20,250,6066
6,Tuesday-WorkingHours.pcap_ISCX.csv,445909,79,78,201,327,24065
7,Wednesday-workingHours.pcap_ISCX.csv,692703,79,78,1008,1586,81909


In [10]:
manifest = {
    "dataset":"CIC-IDS2017",
    "files":[str(x) for x in audit["file"].tolist()],
    "audit":audit.to_dict(orient="records"),
    "integrity":integrity_df.to_dict(orient="records"),
    "note":"Source file is the validation group. Do not mix rows from a holdout file into training."
}
out=META_DIR/"model3_v2_group_manifest.json"
out.write_text(json.dumps(manifest,indent=2,default=str),encoding="utf-8")
print("Saved:",out)

Saved: c:\Users\bimal\OneDrive\Desktop\EVIDENT\models\behavior\metadata\model3_v2_group_manifest.json
